# 07 — DFS Constraint

**Repo:** `decoherence-free-sensing`  
**Lab report:** *Decoherence-Free Subspaces Specify Robust Quantum Sensing*  
**Paper:** Kaubruegger *et al.*, *Physical Review X* **16**, 021052 (2026), DOI: `10.1103/ksyh-mb4s`

Notebook 00 established the leading specification:

\[
\text{common-mode noise}
\rightarrow
\text{DFS constraint}
\rightarrow
\text{DFS entanglement}
\rightarrow
\text{robust differential phase sensing}.
\]

Notebook 07 isolates the constraint itself.

## Engineering statement

A decoherence-free sensor is specified by keeping preparation, evolution, and measurement inside one fixed \(J_z^+\) sector, where common-mode phase noise acts only as a global phase.

The rule is simple:

\[
J_z^+|\psi\rangle=m|\psi\rangle
\quad\Longrightarrow\quad
U_\Phi|\psi\rangle=e^{-i\Phi J_z^+}|\psi\rangle=e^{-i\Phi m}|\psi\rangle.
\]

The phase \(e^{-i\Phi m}\) is global, so it cannot change probabilities inside that sector.


## Notebook outputs

Running this notebook creates:

- `results/figures/07_common_phase_as_global_phase.png`
- `results/figures/07_dfs_sector_ladder.png`
- `results/figures/07_measurement_commutation_rule.png`
- `results/figures/07_inside_vs_outside_dfs.png`
- `results/figures/07_dfs_design_rule_summary.png`
- `results/dfs_constraint_summary.json`
- `results/dfs_constraint_summary.csv`
- `results/decoherence_free_sensing_07_outputs.zip`


In [ ]:
from pathlib import Path
import json
import math
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
SITE = ROOT / "site"

for path in [RESULTS, FIGURES, SITE]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})


## 1. Common phase as a global phase

The common phase is generated by

\[
J_z^+=J_z^A+J_z^B.
\]

If a state is restricted to one eigen-sector of \(J_z^+\), then common-mode phase noise cannot change relative amplitudes inside that state. It contributes only a global phase.


In [ ]:
def add_box(ax, xy, width, height, title, body, title_size=14, body_size=11):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.02,rounding_size=0.035",
        linewidth=1.8,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(x + width/2, y + height*0.66, title,
            ha="center", va="center", fontsize=title_size, fontweight="bold")
    ax.text(x + width/2, y + height*0.34, body,
            ha="center", va="center", fontsize=body_size)


def add_arrow(ax, x0, y0, x1, y1):
    arr = FancyArrowPatch(
        (x0, y0), (x1, y1),
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.8
    )
    ax.add_patch(arr)

fig, ax = plt.subplots(figsize=(13.5, 5.6))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.91, "Common Phase Becomes Global Inside One DFS Sector",
        ha="center", va="center", fontsize=22)

boxes = [
    (0.05, r"common phase", r"$U_\Phi=e^{-i\Phi J_z^+}$"),
    (0.30, r"fixed sector", r"$J_z^+|\psi\rangle=m|\psi\rangle$"),
    (0.55, r"global phase", r"$U_\Phi|\psi\rangle=e^{-i\Phi m}|\psi\rangle$"),
    (0.80, r"probabilities", r"unchanged by $\Phi$"),
]

w, h, y = 0.17, 0.28, 0.43
for x, title, body in boxes:
    add_box(ax, (x, y), w, h, title, body)

for x0, x1 in [(0.22, 0.30), (0.47, 0.55), (0.72, 0.80)]:
    add_arrow(ax, x0, y + h/2, x1, y + h/2)

ax.text(
    0.5, 0.22,
    "Design rule: choose states, operations, and measurements that do not leak information out of the fixed $J_z^+$ sector.",
    ha="center", va="center", fontsize=14
)

path = FIGURES / "07_common_phase_as_global_phase.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 2. \(J_z^+\) sector ladder

For a two-node sensor, computational basis states can be grouped by their total excitation imbalance, represented here as a \(J_z^+\) eigenvalue.

A DFS is not a single state. It is a sector. The central sensing protocols live in a chosen sector and avoid creating coherences between sectors that would be sensitive to common phase noise.


In [ ]:
from itertools import product

# Small illustrative system: 4 atoms total, 2 in node A and 2 in node B.
# Use ↑ = +1/2, ↓ = -1/2. J_z^+ is the total z projection.
n_atoms = 4
states = list(product([1, -1], repeat=n_atoms))
records = []
for bits in states:
    m = sum(bits) / 2
    label = "".join("↑" if b == 1 else "↓" for b in bits[:2]) + " | " + "".join("↑" if b == 1 else "↓" for b in bits[2:])
    records.append((m, label))

sectors = {}
for m, label in records:
    sectors.setdefault(m, []).append(label)

fig, ax = plt.subplots(figsize=(10.5, 7.0))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.94, r"Hilbert Space Splits into Fixed $J_z^+$ Sectors", ha="center", fontsize=21)

ys = np.linspace(0.80, 0.18, len(sorted(sectors.keys(), reverse=True)))
for y, m in zip(ys, sorted(sectors.keys(), reverse=True)):
    labels = sectors[m]
    ax.hlines(y, 0.18, 0.92, linewidth=1.5, alpha=0.55)
    ax.text(0.08, y, rf"$m={m:g}$", ha="center", va="center", fontsize=13, fontweight="bold")
    text = "   ".join(labels)
    if m == 0:
        ax.text(0.55, y, text, ha="center", va="center", fontsize=13,
                bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="black", linewidth=1.5))
        ax.text(0.93, y, "central DFS", ha="left", va="center", fontsize=12)
    else:
        ax.text(0.55, y, text, ha="center", va="center", fontsize=11, alpha=0.75)

ax.text(0.5, 0.07, r"Common phase adds sector-dependent phases $e^{-i\Phi m}$; inside one sector it is global.",
        ha="center", fontsize=13)

path = FIGURES / "07_dfs_sector_ladder.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 3. DFS-compatible measurement rule

To remain insensitive to common phase noise, the measurement should not couple different \(J_z^+\) sectors. A concise rule is

\[
[M,J_z^+]=0.
\]

The paper's two-body observable is compatible with this rule:

\[
M=J_A^+J_B^-+J_A^-J_B^+.
\]

It moves one excitation between the two nodes while preserving the total \(J_z^+\) sector.


In [ ]:
# Build a small matrix demonstration for two atoms: one atom in A and one in B.
# Basis order: |↑↑>, |↑↓>, |↓↑>, |↓↓>
# Jz+ = JzA + JzB. M = σ_A^+ σ_B^- + σ_A^- σ_B^+.

up = np.array([1, 0], dtype=complex)
down = np.array([0, 1], dtype=complex)
sigma_plus = np.array([[0, 1], [0, 0]], dtype=complex)
sigma_minus = np.array([[0, 0], [1, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex) / 2
I = np.eye(2, dtype=complex)

Jz_plus = np.kron(sigma_z, I) + np.kron(I, sigma_z)
M_good = np.kron(sigma_plus, sigma_minus) + np.kron(sigma_minus, sigma_plus)
M_bad = np.kron(np.array([[0,1],[1,0]], dtype=complex), I)  # local sigma_x on A, changes sector

comm_good = M_good @ Jz_plus - Jz_plus @ M_good
comm_bad = M_bad @ Jz_plus - Jz_plus @ M_bad

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.7))

for ax, mat, title in [
    (axes[0], np.abs(comm_good), r"compatible: $[M,J_z^+]=0$"),
    (axes[1], np.abs(comm_bad), r"not compatible: local $\sigma_x^A$ leaks sectors"),
]:
    im = ax.imshow(mat, vmin=0, vmax=max(1.0, np.max(mat)), aspect="equal")
    ax.set_title(title)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(["↑↑", "↑↓", "↓↑", "↓↓"])
    ax.set_yticklabels(["↑↑", "↑↓", "↓↑", "↓↓"])
    ax.set_xlabel("basis state")
    ax.set_ylabel("basis state")

fig.suptitle("Measurement Compatibility Is a Commutation Rule", fontsize=18, y=1.03)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.78, label="absolute commutator entry")

path = FIGURES / "07_measurement_commutation_rule.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 4. Inside versus outside the DFS

The distinction is visible with a two-state superposition.

Inside one sector:

\[
|\psi_{in}\rangle = \frac{|\uparrow\downarrow\rangle + |\downarrow\uparrow\rangle}{\sqrt{2}}.
\]

Both basis states have \(m=0\), so the common phase is global.

Outside one sector:

\[
|\psi_{out}\rangle = \frac{|\uparrow\uparrow\rangle + |\downarrow\downarrow\rangle}{\sqrt{2}}.
\]

The two components have different \(J_z^+\) values, so common phase becomes a relative phase.


In [ ]:
Phi = np.linspace(0, 2*np.pi, 600)

# Coherence-sensitive fringe proxy:
# inside DFS common phase leaves relative phase unchanged => constant.
inside_signal = np.ones_like(Phi)
# outside DFS relative phase changes by 2 Phi between m=+1 and m=-1 components.
outside_signal = np.cos(2 * Phi)

fig, ax = plt.subplots(figsize=(8.4, 5.7))
ax.plot(Phi, inside_signal, linewidth=2.2, label="inside one DFS sector")
ax.plot(Phi, outside_signal, linewidth=2.2, label="superposition across sectors")
ax.set_title("Inside DFS: Common Phase Invisible. Outside DFS: Relative Phase Moves.")
ax.set_xlabel(r"common phase $\Phi$")
ax.set_ylabel("normalized coherence proxy")
ax.set_ylim(-1.12, 1.12)
ax.grid(True, alpha=0.3)
ax.legend()

ax.annotate("global only", xy=(4.8, 1.0), xytext=(4.2, 0.55),
            arrowprops=dict(arrowstyle="->", linewidth=1.2), fontsize=12)
ax.annotate("relative phase", xy=(4.7, np.cos(9.4)), xytext=(3.5, -0.70),
            arrowprops=dict(arrowstyle="->", linewidth=1.2), fontsize=12)

path = FIGURES / "07_inside_vs_outside_dfs.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 5. Design rule summary

A DFS-compatible protocol must satisfy the constraint at every stage:

1. prepare states in one fixed \(J_z^+\) sector,
2. use dynamics that preserve that sector,
3. measure observables that commute with \(J_z^+\),
4. encode the signal with \(J_z^-\), the differential generator.

The leading specification is therefore not a specific state. It is a compatibility rule for the whole sensing protocol.


In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.text(0.5, 0.92, "DFS Design Rule Summary", ha="center", fontsize=23)

steps = [
    (0.05, "Prepare", r"state in one\n$J_z^+$ sector"),
    (0.28, "Evolve", r"operations preserve\n$J_z^+$"),
    (0.51, "Measure", r"observables commute\nwith $J_z^+$"),
    (0.74, "Estimate", r"signal generated by\n$J_z^-$"),
]

w, h, y = 0.18, 0.30, 0.43
for x, title, body in steps:
    add_box(ax, (x, y), w, h, title, body, title_size=15, body_size=12)

for x0, x1 in [(0.23, 0.28), (0.46, 0.51), (0.69, 0.74)]:
    add_arrow(ax, x0, y+h/2, x1, y+h/2)

ax.text(0.5, 0.24,
        r"Common-mode phase $\Phi$ is rejected only when the full protocol respects the DFS constraint.",
        ha="center", fontsize=14)
ax.text(0.5, 0.15,
        r"Then entanglement can be optimized for differential phase $\varphi$ without amplifying shared noise.",
        ha="center", fontsize=14)

path = FIGURES / "07_dfs_design_rule_summary.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 6. Constraint summary table

In [ ]:
summary = pd.DataFrame([
    {
        "item": "leading_specification",
        "value": "Keep preparation, evolution, and measurement inside one fixed J_z^+ sector."
    },
    {
        "item": "common_phase_unitary",
        "value": "U_Phi = exp(-i Phi J_z^+)"
    },
    {
        "item": "dfs_condition",
        "value": "J_z^+ |psi> = m |psi>"
    },
    {
        "item": "global_phase_result",
        "value": "U_Phi |psi> = exp(-i Phi m) |psi>"
    },
    {
        "item": "measurement_rule",
        "value": "[M, J_z^+] = 0"
    },
    {
        "item": "compatible_two_body_observable",
        "value": "M = J_A^+ J_B^- + J_A^- J_B^+"
    },
    {
        "item": "failure_mode",
        "value": "Superpositions across different J_z^+ sectors turn common phase into relative phase."
    },
    {
        "item": "engineering_rule",
        "value": "Reject shared noise first; optimize differential sensitivity second."
    },
])

summary_csv = RESULTS / "dfs_constraint_summary.csv"
summary_json = RESULTS / "dfs_constraint_summary.json"
summary.to_csv(summary_csv, index=False)
summary.to_json(summary_json, orient="records", indent=2)
summary


## 7. Export notebook outputs

This cell creates one zip archive containing all generated figures and DFS-constraint summary files.


In [ ]:
zip_path = RESULTS / "decoherence_free_sensing_07_outputs.zip"

files_to_zip = [
    FIGURES / "07_common_phase_as_global_phase.png",
    FIGURES / "07_dfs_sector_ladder.png",
    FIGURES / "07_measurement_commutation_rule.png",
    FIGURES / "07_inside_vs_outside_dfs.png",
    FIGURES / "07_dfs_design_rule_summary.png",
    summary_csv,
    summary_json,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in files_to_zip:
        if file.exists():
            zf.write(file, file.relative_to(ROOT))

print(f"Wrote: {zip_path}")
zip_path

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))


## 8. Next notebook

Suggested next notebook:

`13_phase_coordinates.ipynb`

Core goal:

- formalize \(\Phi=(\phi_A+\phi_B)/2\) and \(\varphi=(\phi_A-\phi_B)/2\),
- simulate how common-mode noise spreads single-node estimates,
- show how the differential coordinate preserves the signal,
- connect the geometric phase coordinates to the DFS operator constraint.
